# Cardiovascular-Kidney-Metabolic Syndrome — genai-multiturn-messaging-probe

## Access Popular LLMs via Google-Colab-AI Without an API Key

This notebook keeps the original Ollama-in-Colab setup **intact and unchanged**, and adds
two layers on top of it:

1. **RAG** — grounds `llama3.1:8b` in PDF documents (starting with the 2026 AHA/ACC/ADA/ASN
   CKM guideline), extendable to any future PDFs.
2. **Messaging evaluator** — scores the grounded model on simulated CKM patient-portal
   messages loaded from `ckm_patients.json`, each with structured EHR/lab context and an
   expected triage result.

Nothing from the original notebook is removed. `generate_ollama_response()`, the version
probe, and the "capital of France" test all remain and still work; the RAG and evaluator
sit alongside them.

**Layout**

| Part | Sections | Origin |
|---|---|---|
| Ollama setup, model tests | 1–5 | **original, unchanged** |
| RAG over PDFs | 6–11 | new |
| Patient messaging evaluator | 12–16 | new |

> **Runtime:** select a **GPU** runtime (`Runtime > Change runtime type`), then run top to bottom.

---
# Part I — Original notebook (unchanged)

## More memory

Users who have purchased one of Colab's paid plans have access to high-memory VMs when they are available. More powerful GPUs are always offered with high-memory VMs.

You can see how much memory you have available at any time by running the following code cell. If the execution result of running the code cell below is "Not using a high-RAM runtime", then you can enable a high-RAM runtime via `Runtime > Change runtime type` in the menu. Then select High-RAM in the Runtime shape toggle button. After, re-execute the code cell.

In [1]:
%pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 55.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 76.9 MB/s eta 0:00:00
  Cloning https://github.com/googlecolab/kernel_gateway (to revision b134e9945df25c2dcb98ade9129399be10788671) to /tmp/pip-install-lblbp6my/jupyter-kernel-gateway_4eb99e4d19b447b888df93ec7db9fbce
  Running command git clone --filter=blob:none --quiet https://github.com/googlecolab/kernel_gateway /tmp/pip-install-lblbp6my/jupyter-kernel-gateway_4eb99e4d19b447b888df93ec7db9fbce
  Running command git rev-parse -q --verify 'sha^b134e9945df25c2dcb98ade9129399be10788671'
  Running command git fetch -q https://github.com/googlecolab/kernel_gateway b134e9945df25c2dcb98ade9129399be10788671
  Running command git checkout -q b134e9945df25c2dcb98ade9129399be10788671
  Resolved https://github.com/googlecolab/kernel_gateway to commit b134e9945df25c2dcb98ade9129399b

In [2]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 13.6 gigabytes of available RAM

Not using a high-RAM runtime


## Longer runtimes

All Colab runtimes are reset after some period of time (which is faster if the runtime isn't executing code). Colab Pro and Pro+ users have access to longer runtimes than those who use Colab free of charge.

## Background execution

Colab Pro+ users have access to background execution, where notebooks will continue executing even after you've closed a browser tab. This is always enabled in Pro+ runtimes as long as you have compute units available.

### Running Ollama directly in Colab

To run Ollama directly within your Colab environment, you'll need to install it and then pull the desired model. For larger models, ensure you have a GPU runtime enabled (`Runtime > Change runtime type` and select a GPU accelerator).

In [3]:
# 1. Install Ollama in Colab
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

print('Ollama installation script executed.')

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [4]:
# @title Select your desired Ollama model
# You can find more models at https://ollama.com/library
DEFAULT_MODEL = "llama3.1:8b" # @param ["llama2","phi3","mistral","tinyllama","gemma:2b","stable-code","llama3.1","mxbai-embed-large","llama3.1:8b"]

print(f"Selected model: {DEFAULT_MODEL}")

Selected model: llama3.1:8b


After installing Ollama, you need to start the Ollama server. This usually happens automatically upon installation or the first `ollama` command, but explicitly starting it ensures it's running in the background.

In [5]:
import subprocess
import time
import os
import requests

OLLAMA_BINARY_PATH = "/usr/local/bin/ollama"
OLLAMA_HOST = "http://localhost:11434"

print(f"Ensuring a fresh Ollama server instance at {OLLAMA_HOST}...")

# Terminate any existing Ollama processes to prevent port conflicts
try:
    # Find PIDs of processes listening on port 11434 and kill them
    pids_output = subprocess.check_output(f"lsof -i tcp:11434 | grep LISTEN | awk '{{print $2}}'", shell=True, text=True)
    pids = [int(p) for p in pids_output.strip().split('\n') if p]
    if pids:
        print(f"Killing existing Ollama process(es): {pids}")
        for pid in pids:
            os.kill(pid, 9) # Send SIGKILL
            time.sleep(1) # Give it a moment to die
except subprocess.CalledProcessError:
    # This occurs if lsof finds nothing or grep doesn't match 'LISTEN'
    pass # No existing processes to kill
except Exception as e:
    print(f"Warning: Error during attempt to kill old Ollama processes: {e}")

# Try to start Ollama server using subprocess.Popen for background execution
try:
    ollama_process = subprocess.Popen(
        [OLLAMA_BINARY_PATH, 'serve'],
        stdout=subprocess.DEVNULL, # Redirect stdout to DEVNULL for less clutter
        stderr=subprocess.DEVNULL, # Redirect stderr to DEVNULL for less clutter
    )
    print(f"Ollama server process started (PID: {ollama_process.pid}).")

    # Wait for Ollama to be ready
    ready = False
    start_time = time.time()
    timeout = 90 # Increase timeout for Ollama to spin up
    print("Waiting for Ollama server to become ready...")
    while not ready and (time.time() - start_time < timeout):
        try:
            # Use requests for a more robust HTTP check
            # Ollama's root endpoint often returns 404 but indicates server is up
            response = requests.get(f'{OLLAMA_HOST}', timeout=5)
            if response.status_code in [200, 404]: # 404 is fine for base URL of Ollama
                ready = True
        except requests.exceptions.ConnectionError:
            pass # Keep trying
        except requests.exceptions.Timeout:
            pass # Keep trying
        time.sleep(5) # Check every 5 seconds

    if ready:
        print("Ollama server is ready and accessible.")
    else:
        print(f"Error: Ollama server did not become ready within {timeout} seconds.")

except FileNotFoundError:
    print(f"Error: Ollama binary not found at {OLLAMA_BINARY_PATH}. Please ensure Ollama was installed correctly in the previous step.")
except Exception as e:
    print(f"Error starting Ollama server: {e}")

print(f'OLLAMA_HOST is set to: {OLLAMA_HOST}')

Ensuring a fresh Ollama server instance at http://localhost:11434...
Ollama server process started (PID: 7787).
Waiting for Ollama server to become ready...
Ollama server is ready and accessible.
OLLAMA_HOST is set to: http://localhost:11434


In [6]:
# Pull the selected model
# This can take a while depending on the model size and your internet connection.
!{OLLAMA_BINARY_PATH} pull {DEFAULT_MODEL}

print(f'Model {DEFAULT_MODEL} pulled.')


Model llama3.1:8b pulled.


### 4. Verify connection and model (Example)

Now your Colab environment has Ollama running locally within it, and you can use `urllib.request` or any HTTP client to interact with it.

In [7]:
import urllib.request
import urllib.error
import json

# Example: Check if Ollama server is reachable and list models
try:
    with urllib.request.urlopen(f'{OLLAMA_HOST}/api/tags') as response:
        data = json.loads(response.read().decode())
        print(f"Successfully connected to Ollama! Available models: {data.get('models', [])}")
except urllib.error.URLError as e:
    print(f"Could not connect to Ollama: {e}")
    print("Please ensure the Ollama server started correctly in this Colab session.")

Successfully connected to Ollama! Available models: [{'name': 'llama3.1:8b', 'model': 'llama3.1:8b', 'modified_at': '2026-08-02T21:46:44.482814348Z', 'size': 4920753328, 'digest': '46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 4096}, 'capabilities': ['completion', 'tools']}, {'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'modified_at': '2026-08-02T21:36:57.156416869Z', 'size': 274302450, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16', 'context_length': 2048, 'embedding_length': 768}, 'capabilities': ['embedding']}]


In [8]:
import requests, json

resp = requests.get(f'{OLLAMA_HOST}/api/tags')
models = [m['name'] for m in resp.json().get('models', [])]
print(models)  # should include 'llama3.1:8b'

requests.post(f"{OLLAMA_HOST}/api/generate",
              json={"model": DEFAULT_MODEL, "keep_alive": 0})

['llama3.1:8b', 'nomic-embed-text:latest']


<Response [200]>

### 5. Test the Model

In [9]:
import requests
import json

def generate_ollama_response(prompt, model=DEFAULT_MODEL, host=OLLAMA_HOST):
    url = f'{host}/api/generate'
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False, # Get the full response at once
        "options": {"temperature": 0, "seed": 42}
    }
    headers = {'Content-Type': 'application/json'}

    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=60)
        response.raise_for_status() # Raise an exception for HTTP errors
        return response.json()['response'].strip()
    except requests.exceptions.RequestException as e:
        print(f"Error generating response: {e}")
        return None

# Test 1: Ask for the model's version
print(f"Querying '{DEFAULT_MODEL}' for its version...")
version_prompt = "What is your version?"
model_version_response = generate_ollama_response(version_prompt)
if model_version_response:
    print(f"Response for 'What is your version?':\n{model_version_response}\n")

# Test 2: Ask a general knowledge question
print(f"Querying '{DEFAULT_MODEL}' for 'What is the capital of France?'...")
capital_prompt = "What is the capital of France?"
capital_response = generate_ollama_response(capital_prompt)
if capital_response:
    print(f"Response for 'What is the capital of France?':\n{capital_response}")

Querying 'llama3.1:8b' for its version...
Response for 'What is your version?':
This conversation has just begun. I'm happy to chat with you, but I don't have a "version" in the sense that I'm not a physical product or software release. I exist as a cloud-based service designed to provide information and assist with tasks.

If you could provide more context about what you're referring to, I'd be happy to try and help!

Querying 'llama3.1:8b' for 'What is the capital of France?'...
Response for 'What is the capital of France?':
The capital of France is Paris.


---
# Part II — RAG layer (new)

Everything above still works exactly as before. `generate_ollama_response()` remains
available for ungrounded queries, which is useful as a **baseline**: the evaluator in
Part III can run with RAG on or off to measure what retrieval actually contributes.

## 6. RAG configuration

The single place to tune the pipeline.

- **`EMBED_MODEL`** — `nomic-embed-text` (768-d) is fast. `mxbai-embed-large` (1024-d, already
  in your model dropdown) gives better recall. *Changing this invalidates the index — delete
  `CHROMA_DIR` and re-ingest.*
- **`CHROMA_DIR`** — `/content` is **wiped when the session ends**. Point at a mounted Drive
  path to persist the index and skip re-embedding each session.
- **`USE_RERANKER`** — reranking is the highest-leverage quality lever after chunking.

In [10]:
CHAT_MODEL  = DEFAULT_MODEL       # reuse the model selected in the @param cell above
EMBED_MODEL = "nomic-embed-text"  # 768-d, fast; "mxbai-embed-large" (1024-d) for higher recall

# Storage. /content is ephemeral on Colab; use a Drive path to persist across sessions.
CHROMA_DIR = "/content/ckm_rag/chroma"
COLLECTION = "ckm_corpus"

# Chunking
CHUNK_TARGET_CHARS  = 1200   # ~300 tokens; balances context against retrieval precision
CHUNK_OVERLAP_CHARS = 200    # carries context across chunk boundaries

# Retrieval
RETRIEVE_K   = 12            # candidates pulled from the vector store
USE_RERANKER = True          # cross-encoder rerank of candidates (recommended)
FINAL_K      = 5             # chunks actually shown to the LLM after reranking
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Generation. temperature 0 + fixed seed keeps evaluation runs reproducible.
GEN_OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 8192}

print("RAG config loaded.")

RAG config loaded.


## 7. Install RAG dependencies

| Package | Role |
|---|---|
| `pymupdf` | PDF text extraction with page numbers |
| `chromadb` | Persistent vector store |
| `sentence-transformers` | Cross-encoder reranker (skipped if `USE_RERANKER = False`) |

In [11]:
#%pip -q install pymupdf chromadb
#if USE_RERANKER:
#    %pip -q install sentence-transformers

# The embedding model runs on the same Ollama server started above.
!{OLLAMA_BINARY_PATH} pull {EMBED_MODEL}
print(f"Embedding model {EMBED_MODEL} pulled.")


Embedding model nomic-embed-text pulled.


## 8. Ollama helpers for RAG

`generate_ollama_response()` from Section 5 stays as-is. These two additions cover what RAG
needs and it does not: **batched embeddings**, and a **chat-endpoint** call that accepts a
separate system prompt (`/api/generate` takes a single flat prompt, which makes the grounding
instruction harder to keep separate from the retrieved context).

In [12]:
def embed_texts(texts, model=EMBED_MODEL, host=OLLAMA_HOST, batch_size=64):
    """Return a list of embedding vectors for a list of strings (batched)."""
    if isinstance(texts, str):
        texts = [texts]
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        try:
            # /api/embed is the current batch endpoint
            r = requests.post(f"{host}/api/embed", json={"model": model, "input": batch}, timeout=120)
            r.raise_for_status()
            vectors.extend(r.json()["embeddings"])
        except Exception:
            # Fall back to the older one-at-a-time endpoint
            for t in batch:
                r = requests.post(f"{host}/api/embeddings", json={"model": model, "prompt": t}, timeout=120)
                r.raise_for_status()
                vectors.append(r.json()["embedding"])
    return vectors


def chat_ollama_response(system, user, model=CHAT_MODEL, host=OLLAMA_HOST, options=None):
    """Single-turn chat completion with a separate system prompt, via /api/chat."""
    payload = {
        "model": model,
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "stream": False,
        "options": options or GEN_OPTIONS,
    }
    try:
        r = requests.post(f"{host}/api/chat", json=payload, timeout=300)
        r.raise_for_status()
        return r.json()["message"]["content"].strip()
    except requests.exceptions.RequestException as e:
        print(f"Error generating chat response: {e}")
        return None


# Smoke test both helpers before spending time on ingestion.
print("embedding dim:", len(embed_texts(["test"])[0]))
print("chat:", chat_ollama_response("Reply in exactly one word.", "Say OK."))

embedding dim: 768
chat: OK.


## 9. PDF loading and chunking

`_clean()` handles two artifacts present in the CKM guideline PDF: soft hyphens
(`develop\u00adment`) and line-break hyphenation (`develop-\nment`). Left in, both corrupt
the embedded tokens and degrade retrieval.

**Chunking is done per page**, which keeps page numbers exact for citation — necessary when a
reader has to verify a recommendation against the source guideline.

In [13]:
import os, re, hashlib
import fitz  # PyMuPDF


def _clean(text):
    """Normalize whitespace, drop soft hyphens, de-hyphenate line breaks."""
    text = text.replace("\u00ad", "")                 # soft hyphen
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)   # "develop-\nment" -> "development"
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_pdf_pages(path):
    """Return [(page_number, cleaned_text), ...] for pages that contain text."""
    doc = fitz.open(path)
    pages = []
    for i, page in enumerate(doc):
        txt = _clean(page.get_text("text"))
        if txt:
            pages.append((i + 1, txt))
    doc.close()
    return pages


def chunk_page(text, target=CHUNK_TARGET_CHARS, overlap=CHUNK_OVERLAP_CHARS):
    """Pack paragraphs into ~target-char chunks; hard-split any oversized paragraph."""
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, buf = [], ""
    for p in paras:
        if len(buf) + len(p) + 1 <= target:
            buf = (buf + "\n" + p).strip()
        else:
            if buf:
                chunks.append(buf)
            if len(p) > target:
                start = 0
                while start < len(p):
                    chunks.append(p[start:start + target])
                    start += max(1, target - overlap)
                buf = ""
            else:
                buf = p
    if buf:
        chunks.append(buf)
    return chunks


def build_chunks(path):
    """Turn one PDF into a list of chunk records with citation metadata."""
    source = os.path.basename(path)
    records = []
    for page_no, page_text in load_pdf_pages(path):
        for idx, chunk in enumerate(chunk_page(page_text)):
            uid = hashlib.sha1(f"{source}|{page_no}|{idx}|{chunk[:80]}".encode()).hexdigest()
            records.append({
                "id": uid,
                "text": chunk,
                "metadata": {"source": source, "page": page_no, "chunk": idx},
            })
    return records

## 10. Vector store and idempotent ingestion

Chunk IDs are content-derived SHA-1 hashes, so `ingest_documents()` skips anything already
stored rather than re-embedding it. Re-running is cheap and never duplicates. **This is the
extension point** — adding future PDFs is one call.

Cosine distance is set explicitly; Chroma defaults to L2, a poorer fit for text embeddings.

In [14]:
import chromadb

os.makedirs(CHROMA_DIR, exist_ok=True)
_client = chromadb.PersistentClient(path=CHROMA_DIR)
_collection = _client.get_or_create_collection(name=COLLECTION, metadata={"hnsw:space": "cosine"})


def ingest_documents(paths, verbose=True):
    """Ingest one or more PDFs into the vector store. Idempotent."""
    if isinstance(paths, str):
        paths = [paths]

    all_records = []
    for path in paths:
        recs = build_chunks(path)
        all_records.extend(recs)
        if verbose:
            print(f"  {os.path.basename(path)}: {len(recs)} chunks")

    ids = [r["id"] for r in all_records]
    existing = set()
    for i in range(0, len(ids), 500):
        existing.update(_collection.get(ids=ids[i:i + 500]).get("ids", []))
    new = [r for r in all_records if r["id"] not in existing]

    if not new:
        print(f"Nothing new to embed. Collection size: {_collection.count()}")
        return

    if verbose:
        print(f"Embedding {len(new)} new chunks ({len(all_records) - len(new)} already present)...")
    vectors = embed_texts([r["text"] for r in new])

    for i in range(0, len(new), 500):
        batch = new[i:i + 500]
        _collection.upsert(
            ids=[r["id"] for r in batch],
            embeddings=vectors[i:i + 500],
            documents=[r["text"] for r in batch],
            metadatas=[r["metadata"] for r in batch],
        )
    print(f"Ingested. Collection size: {_collection.count()}")

## 11. Retrieval, reranking, and grounded answers

Two-stage retrieval: embedding similarity casts a wide net (`RETRIEVE_K`), then a
cross-encoder reorders by true query-passage relevance and keeps `FINAL_K`. The bi-encoder
compares query and passage encoded *independently*; the cross-encoder reads both together —
slower, but markedly more accurate. Running it over a shortlist is what makes it affordable.

In [15]:
_reranker = None


def _get_reranker():
    """Lazy-load the cross-encoder so CPU-only sessions can skip the cost."""
    global _reranker
    if _reranker is None:
        from sentence_transformers import CrossEncoder
        print(f"Loading reranker {RERANK_MODEL} ...")
        _reranker = CrossEncoder(RERANK_MODEL)
    return _reranker


def retrieve(question, k=RETRIEVE_K, final_k=FINAL_K, use_reranker=USE_RERANKER):
    """Return the top chunks for a question as a list of dicts."""
    qvec = embed_texts([question])[0]
    res = _collection.query(
        query_embeddings=[qvec],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    hits = [
        {"text": d, "metadata": m, "distance": dist}
        for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]

    if use_reranker and hits:
        scores = _get_reranker().predict([(question, h["text"]) for h in hits])
        for h, s in zip(hits, scores):
            h["rerank_score"] = float(s)
        hits.sort(key=lambda h: h["rerank_score"], reverse=True)

    return hits[:final_k]


def _format_context(hits):
    """Render retrieved chunks as [S#] blocks plus a reference list."""
    blocks, refs = [], []
    for i, h in enumerate(hits, 1):
        m = h["metadata"]
        tag = f"S{i}"
        blocks.append(f"[{tag}] (source: {m['source']}, p.{m['page']})\n{h['text']}")
        refs.append(f"[{tag}] {m['source']}, p.{m['page']}")
    return "\n\n".join(blocks), refs


RAG_SYSTEM_PROMPT = (
    "You are a careful clinical-informatics assistant. Answer ONLY from the "
    "provided context. If the answer is not in the context, say you cannot find "
    "it in the provided documents - do not use outside knowledge and do not "
    "guess. Cite every claim with the bracketed source tags shown in the context "
    "(e.g., [S1], [S2]). Be concise and precise; this may inform clinical "
    "decisions and must be verifiable against the sources."
)


def answer(question, show_sources=True, **retrieval_kwargs):
    """Retrieve, then generate a grounded, cited answer."""
    hits = retrieve(question, **retrieval_kwargs)
    if not hits:
        return "No documents have been ingested yet. Call ingest_documents([...]) first."

    context, refs = _format_context(hits)
    user_prompt = f"Context:\n\n{context}\n\nQuestion: {question}\n\nAnswer (with [S#] citations):"
    reply = chat_ollama_response(RAG_SYSTEM_PROMPT, user_prompt)

    if show_sources:
        reply += "\n\nRetrieved sources:\n" + "\n".join(refs)
    return reply

### Ingest the guideline corpus

Upload the PDF via the Colab **Files** pane (or mount Drive), then set the path below.
Expected for the CKM guideline: **109 pages → ~679 chunks**. The first ingestion takes a few
minutes; re-running afterwards is near-instant.

In [16]:
DOCS = [
    "/content/ndumele-et-al-2026-ckm.pdf",
]

# Fail early with a clear message rather than a confusing traceback deep inside fitz.
for p in DOCS:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Not found: {p}\nUpload it via the Files pane and fix the path above.")

ingest_documents(DOCS)

  ndumele-et-al-2026-ckm.pdf: 679 chunks
Embedding 679 new chunks (0 already present)...
Ingested. Collection size: 679


### Grounded QA, and the refusal check

A grounded system's most important behaviour is admitting ignorance. The second query asks
something the guideline does not cover — if the model answers it fluently anyway, grounding
is not working.

In [17]:
print(answer("What are the first-line therapies for patients with CKD and type 2 diabetes?"))
print("\n" + "=" * 80 + "\n")
print(answer("What is the recommended treatment protocol for a fractured tibia?"))

Loading reranker cross-encoder/ms-marco-MiniLM-L-6-v2 ...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

The first-line therapies for patients with CKD and type 2 diabetes are:

* Renin-angiotensin system inhibitors (RASi) [S1]
* Sodium–glucose cotransporter-2 inhibitors (SGLT2i) [S1, S3]

These therapies improve both kidney and cardiovascular outcomes in patients with CKD and type 2 diabetes.

Retrieved sources:
[S1] ndumele-et-al-2026-ckm.pdf, p.53
[S2] ndumele-et-al-2026-ckm.pdf, p.42
[S3] ndumele-et-al-2026-ckm.pdf, p.48
[S4] ndumele-et-al-2026-ckm.pdf, p.23
[S5] ndumele-et-al-2026-ckm.pdf, p.25


I cannot provide medical advice. If you are looking for information on treating a fractured tibia, I suggest consulting a medical professional or a reliable health resource. Is there anything else I can help you with?

Retrieved sources:
[S1] ndumele-et-al-2026-ckm.pdf, p.38
[S2] ndumele-et-al-2026-ckm.pdf, p.89
[S3] ndumele-et-al-2026-ckm.pdf, p.36
[S4] ndumele-et-al-2026-ckm.pdf, p.56
[S5] ndumele-et-al-2026-ckm.pdf, p.28


### Inspect retrieval on its own

Retrieval quality caps answer quality — a wrong answer is usually a retrieval failure, not a
generation failure. Look at what came back before blaming the model.

In [18]:
for h in retrieve("nonsteroidal mineralocorticoid receptor antagonist albuminuria potassium"):
    m = h["metadata"]
    score = h.get("rerank_score")
    score_str = f"rerank={score:.3f}" if score is not None else f"dist={h['distance']:.3f}"
    print(f"[{m['source']} p.{m['page']}] {score_str}")
    print(h["text"][:300].replace("\n", " "), "...\n")

[ndumele-et-al-2026-ckm.pdf p.42] rerank=4.330
 peptide 1; nsMRA, nonsteroidal mineralocorticoid receptor antagonist; RASi; reninangiotensin system inhibitors; SGLT2i, sodium–glucose cotransporter-2 inhibitors; T2D, type 2 diabetes; and UACR, urine albumin-creatinine ratio. Downloaded from http://ahajournals.org by on July 2, 2026 ...

[ndumele-et-al-2026-ckm.pdf p.8] rerank=4.008
MRA nonsteroidal mineralocorticoid receptor antagonist NT-proBNP N-terminal prohormone of B-type natriuretic peptide OR odds ratio OSA obstructive sleep apnea PAD peripheral artery disease PREVENT Predicting Risk of Cardiovascular Disease Events QALY quality-adjusted life-year RAASi renin-angiotensi ...

[ndumele-et-al-2026-ckm.pdf p.60] rerank=0.348
 ejection fraction; HFpEF, heart failure with preserved ejection fraction; HFrEF, heart failure with reduced ejection fraction; nsMRA, nonsteroidal mineralocorticoid receptor antagonist; SGLT2i, sodium–glucose cotransporter-2 inhibitors; and T2D, type 2 diabetes.

---
# Part III — Patient messaging evaluator (new)

## 12. The patient corpus

`ckm_patients.json` holds simulated CKM patients, each with structured EHR context (problems,
medications, vitals, labs) and ambiguous patient-portal messages. Every message carries an
`expected` block for automated scoring.

**Schema**

```
patients[]
  patient_id, demographics, ckm_stage, conditions[], medications[], vitals, labs
  messages[]
    message_id, sent_at, text
    expected:
      triage          one of EMERGENT | URGENT | ROUTINE | SELF_CARE   <- scored
      ambiguity       low | moderate | high                            <- stratifier
      requires_ehr    true if free text alone is insufficient          <- stratifier
      rationale_tags  clinical concepts that should drive the decision
      must_reference  terms the rationale should mention               <- scored (recall)
      key_action      reference answer for human review
      probe_note      why this item is in the corpus
```

**The design point.** `CKM-001-M1` and `CKM-002-M1` are near-identical in free text ("tingling
in my hands, legs feel heavy, probably just getting old") but differ by **two triage levels**,
because CKM-001 has K+ 5.4 on finerenone + lisinopril at eGFR 48 while CKM-002 has K+ 4.1,
eGFR 88, and no RASi or MRA. Pairs like this are what separate EHR-grounded reasoning from
pattern matching on symptom words.

All data is synthetic. No real patient information.

In [19]:
import json

PATIENTS_PATH = "/content/ckm_patients.json"   # upload via the Files pane

if not os.path.exists(PATIENTS_PATH):
    raise FileNotFoundError(f"Not found: {PATIENTS_PATH}\nUpload ckm_patients.json via the Files pane.")

with open(PATIENTS_PATH) as f:
    CORPUS = json.load(f)

TRIAGE_RANK = {lvl["code"]: lvl["rank"] for lvl in CORPUS["triage_levels"]}
TRIAGE_DEFS = "\n".join(f"- {l['code']}: {l['definition']}" for l in CORPUS["triage_levels"])

# Flatten to (patient, message) pairs for iteration.
CASES = [(p, m) for p in CORPUS["patients"] for m in p["messages"]]

print(f"{len(CORPUS['patients'])} patients, {len(CASES)} messages")
print(f"\nTriage levels:\n{TRIAGE_DEFS}")

6 patients, 11 messages

Triage levels:
- EMERGENT: Possible life- or organ-threatening process. Patient should seek emergency care now (911/ED).
- URGENT: Requires clinical contact within 24 hours. Same-day nurse or clinician review, possible medication hold or titration.
- ROUTINE: Needs a clinical response but not urgently. Handle at next business day or schedule a visit.
- SELF_CARE: Educational or informational. Can be answered with guidance; no clinical escalation needed.


## 13. Rendering EHR context

The model sees the structured data as a compact text block. `format_ehr()` is the **ablation
switch** for the probe's central question: calling `triage_message(..., use_ehr=False)`
withholds it, so the same message can be scored with and without structured context and the
difference attributed to the EHR.

In [20]:
def format_ehr(patient):
    """Render a patient's structured EHR context as a compact text block."""
    d = patient["demographics"]
    lines = [
        f"Patient ID: {patient['patient_id']}",
        f"Age/Sex: {d['age']} {d['sex']}",
        f"CKM stage: {patient.get('ckm_stage', 'not staged')}",
        f"Problems: {', '.join(patient['conditions']) if patient['conditions'] else 'none recorded'}",
    ]

    if patient["medications"]:
        meds = "; ".join(f"{m['name']} {m['dose']} ({m['class']}, started {m['started']})"
                         for m in patient["medications"])
    else:
        meds = "none recorded"
    lines.append(f"Medications: {meds}")

    v = patient["vitals"]
    lines.append(
        f"Vitals ({v['date']}): BP {v['bp']}, HR {v['heart_rate']}, "
        f"weight {v['weight_lb']} lb, BMI {v['bmi']}"
    )

    labs = "; ".join(f"{k} {lab['value']} {lab['unit']} ({lab['date']})"
                     for k, lab in patient["labs"].items())
    lines.append(f"Labs: {labs}")

    return "\n".join(lines)


print(format_ehr(CORPUS["patients"][0]))

Patient ID: CKM-001
Age/Sex: 62 female
CKM stage: Stage 3
Problems: Type 2 diabetes mellitus, Chronic kidney disease G3a/A3, Hypertension, Obesity
Medications: Lisinopril 20 mg daily (RASi, started 2023-04-11); Empagliflozin 10 mg daily (SGLT2i, started 2025-09-02); Finerenone 10 mg daily (nonsteroidal MRA, started 2026-06-18); Metformin 1000 mg twice daily (biguanide, started 2019-02-20); Atorvastatin 40 mg daily (statin, started 2021-07-30)
Vitals (2026-06-15): BP 138/84 mmHg, HR 72, weight 198 lb, BMI 34.1
Labs: eGFR 48 mL/min/1.73m2 (2026-06-15); UACR 320 mg/g (2026-06-15); potassium 5.4 mEq/L (2026-07-02); hba1c 7.8 % (2026-06-15); ldl_c 88 mg/dL (2026-06-15)


## 14. The triage function

Assembles patient EHR context + retrieved guideline passages + the message, and requests a
**strict JSON** verdict so scoring is deterministic rather than a prose-parsing exercise.

Three switches, each isolating one contribution:

| Switch | Off means | Measures |
|---|---|---|
| `use_ehr=False` | structured data withheld | what the EHR contributes |
| `use_rag=False` | no guideline passages retrieved | what retrieval contributes |
| both off | free text into the bare model | the original ungrounded baseline |

In [21]:
TRIAGE_SYSTEM = (
    "You are a clinical triage assistant reviewing patient-portal messages for a "
    "cardiovascular-kidney-metabolic (CKM) care team. Weigh the patient's structured EHR "
    "data as heavily as the words of the message: patients routinely minimize symptoms, "
    "offer benign explanations, and bury red flags in casual language. Ground clinical "
    "reasoning in the provided guideline context and cite it with [S#] tags where it "
    "applies. Respond with a single JSON object and nothing else - no preamble, no "
    "markdown fences."
)

TRIAGE_TEMPLATE = """Triage levels:
{triage_defs}

Patient EHR context:
{ehr}

Guideline context:
{context}

Patient portal message (sent {sent_at}):
\"\"\"{message}\"\"\"

Return a JSON object with exactly these keys:
{{
  "triage": one of {codes},
  "red_flags": [short strings, the specific findings driving the decision, [] if none],
  "rationale": "2-3 sentences citing EHR values and [S#] guideline tags",
  "recommended_action": "one sentence, what should happen next"
}}"""


def _parse_json_response(raw):
    """Extract a JSON object from a model response, tolerating fences and stray prose."""
    if not raw:
        return None
    text = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Fall back to the outermost brace-delimited span.
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            return None
    return None


def triage_message(patient, message, use_ehr=True, use_rag=True):
    """Run one message through the model. Returns the parsed verdict plus provenance."""
    codes = list(TRIAGE_RANK)

    if use_rag:
        # Retrieve on the message text plus the problem list: the message alone is often
        # too vague to retrieve the right guideline section.
        query = message["text"] + " " + " ".join(patient["conditions"])
        hits = retrieve(query)
        context, refs = _format_context(hits)
    else:
        hits, refs = [], []
        context = "(no guideline context provided)"

    ehr = format_ehr(patient) if use_ehr else "(structured EHR data withheld)"

    user = TRIAGE_TEMPLATE.format(
        triage_defs=TRIAGE_DEFS,
        ehr=ehr,
        context=context,
        sent_at=message["sent_at"],
        message=message["text"],
        codes=codes,
    )

    raw = chat_ollama_response(TRIAGE_SYSTEM, user)
    parsed = _parse_json_response(raw)

    return {
        "message_id": message["message_id"],
        "patient_id": patient["patient_id"],
        "raw": raw,
        "parsed": parsed,
        "retrieved": refs,
        "use_ehr": use_ehr,
        "use_rag": use_rag,
    }


# Single-case demo: the harder half of the paired probe item.
_p = CORPUS["patients"][0]
_m = _p["messages"][0]
_r = triage_message(_p, _m)
print("MESSAGE:", _m["text"], "\n")
print("EXPECTED:", _m["expected"]["triage"], "|", _m["expected"]["key_action"], "\n")
print("MODEL:", json.dumps(_r["parsed"], indent=2) if _r["parsed"] else _r["raw"])

MESSAGE: Hi, I've been getting a weird tingling in my hands and my legs feel kind of heavy going up the stairs. I'm sure it's nothing, probably just getting older. Do I need to do anything about it? 

EXPECTED: URGENT | Same-day potassium and creatinine check; consider holding finerenone pending result. Paresthesia and muscle weakness on RASi + nonsteroidal MRA with K+ 5.4 and eGFR 48 are consistent with worsening hyperkalemia. 

MODEL: {
  "triage": "URGENT",
  "red_flags": [
    "tingling in hands",
    "heavy legs going up stairs"
  ],
  "rationale": "The patient's symptoms of tingling and heavy legs may indicate potential complications related to CKD or diabetes, warranting urgent clinical contact. According to [S3], referral to an endocrinologist is recommended for further management of hyperglycemia if HbA1c levels are high (HbA1c >10.0%), which in this case is 7.8%. Additionally, the patient's CKD stage and eGFR value suggest a need for closer monitoring.",
  "recommended_action

## 15. Scoring

Four metrics, because plain accuracy hides what matters clinically:

| Metric | Why |
|---|---|
| **Exact accuracy** | headline agreement with the expected label |
| **Under-triage rate** | predicted *less* severe than expected. The dangerous error — a missed EMERGENT is not the same failure as a needless escalation, and accuracy alone treats them identically |
| **Over-triage rate** | predicted more severe. Drives alert fatigue rather than harm |
| **Severity distance** | mean ordinal gap; separates near-misses from wild misses |
| **Reference recall** | fraction of `must_reference` terms appearing in the rationale. Catches a right label reached for the wrong reason |

`severity_distance` uses the ranks declared in the JSON, so adding or reordering triage levels
needs no code change.

In [22]:
import pandas as pd


def score_one(expected, result):
    """Score a single triage result against its expected block."""
    parsed = result["parsed"]
    exp_code = expected["triage"]

    if not parsed or parsed.get("triage") not in TRIAGE_RANK:
        # Unparseable or invalid label is a failure, not a missing value - keep it in the
        # denominator so a model that emits garbage cannot score well by abstaining.
        return {
            "predicted": parsed.get("triage") if parsed else None,
            "expected": exp_code,
            "correct": False,
            "parse_ok": parsed is not None,
            "severity_distance": None,
            "under_triage": None,
            "over_triage": None,
            "reference_recall": 0.0,
        }

    pred_code = parsed["triage"]
    dist = TRIAGE_RANK[pred_code] - TRIAGE_RANK[exp_code]

    # Reference recall over the free-text fields the model produced.
    blob = " ".join([
        parsed.get("rationale", ""),
        parsed.get("recommended_action", ""),
        " ".join(parsed.get("red_flags", []) or []),
    ]).lower()
    terms = expected["must_reference"]
    hits = sum(1 for t in terms if t.lower() in blob)

    return {
        "predicted": pred_code,
        "expected": exp_code,
        "correct": pred_code == exp_code,
        "parse_ok": True,
        "severity_distance": dist,
        "under_triage": dist < 0,
        "over_triage": dist > 0,
        "reference_recall": hits / len(terms) if terms else None,
    }


def evaluate(cases=CASES, use_ehr=True, use_rag=True, verbose=True):
    """Run every case and return a per-message DataFrame."""
    rows = []
    for i, (patient, message) in enumerate(cases, 1):
        if verbose:
            print(f"[{i}/{len(cases)}] {message['message_id']} ...", end=" ")
        result = triage_message(patient, message, use_ehr=use_ehr, use_rag=use_rag)
        scored = score_one(message["expected"], result)
        if verbose:
            mark = "OK  " if scored["correct"] else "MISS"
            print(f"{mark} expected={scored['expected']} predicted={scored['predicted']}")
        rows.append({
            "message_id": message["message_id"],
            "patient_id": patient["patient_id"],
            "ambiguity": message["expected"]["ambiguity"],
            "requires_ehr": message["expected"]["requires_ehr"],
            "use_ehr": use_ehr,
            "use_rag": use_rag,
            **scored,
            "rationale": (result["parsed"] or {}).get("rationale", ""),
            "recommended_action": (result["parsed"] or {}).get("recommended_action", ""),
            "retrieved": "; ".join(result["retrieved"]),
        })
    return pd.DataFrame(rows)


def summarize(df, label=""):
    """Aggregate a results frame into headline metrics."""
    n = len(df)
    return {
        "condition": label,
        "n": n,
        "accuracy": round(df["correct"].mean(), 3),
        "under_triage_rate": round(df["under_triage"].fillna(False).mean(), 3),
        "over_triage_rate": round(df["over_triage"].fillna(False).mean(), 3),
        "mean_abs_severity_distance": round(df["severity_distance"].abs().mean(), 3),
        "reference_recall": round(df["reference_recall"].dropna().mean(), 3),
        "parse_failure_rate": round(1 - df["parse_ok"].mean(), 3),
    }

### Run the full evaluation

In [23]:
results_full = evaluate(use_ehr=True, use_rag=True)

display(results_full[["message_id", "expected", "predicted", "correct",
                      "severity_distance", "reference_recall", "ambiguity", "requires_ehr"]])

print("\nSummary:", json.dumps(summarize(results_full, "EHR + RAG"), indent=2))

[1/11] CKM-001-M1 ... MISS expected=URGENT predicted=ROUTINE
[2/11] CKM-001-M2 ... OK   expected=ROUTINE predicted=ROUTINE
[3/11] CKM-002-M1 ... OK   expected=ROUTINE predicted=ROUTINE
[4/11] CKM-002-M2 ... MISS expected=ROUTINE predicted=URGENT
[5/11] CKM-003-M1 ... OK   expected=URGENT predicted=URGENT
[6/11] CKM-003-M2 ... MISS expected=EMERGENT predicted=URGENT
[7/11] CKM-004-M1 ... OK   expected=URGENT predicted=URGENT
[8/11] CKM-004-M2 ... MISS expected=EMERGENT predicted=URGENT
[9/11] CKM-005-M1 ... OK   expected=ROUTINE predicted=ROUTINE
[10/11] CKM-005-M2 ... MISS expected=SELF_CARE predicted=ROUTINE
[11/11] CKM-006-M1 ... MISS expected=URGENT predicted=ROUTINE


,message_id,expected,predicted,correct,severity_distance,reference_recall,ambiguity,requires_ehr
0,CKM-001-M1,URGENT,ROUTINE,False,-1,0.0,high,True
1,CKM-001-M2,ROUTINE,ROUTINE,True,0,1.0,low,False
2,CKM-002-M1,ROUTINE,ROUTINE,True,0,0.0,high,True
3,CKM-002-M2,ROUTINE,URGENT,False,1,1.0,moderate,False
4,CKM-003-M1,URGENT,URGENT,True,0,0.0,high,True
5,CKM-003-M2,EMERGENT,URGENT,False,-1,0.0,high,True
6,CKM-004-M1,URGENT,URGENT,True,0,1.0,moderate,True
7,CKM-004-M2,EMERGENT,URGENT,False,-1,1.0,moderate,False
8,CKM-005-M1,ROUTINE,ROUTINE,True,0,1.0,moderate,True
9,CKM-005-M2,SELF_CARE,ROUTINE,False,1,1.0,low,False



Summary: {
  "condition": "EHR + RAG",
  "n": 11,
  "accuracy": 0.455,
  "under_triage_rate": 0.364,
  "over_triage_rate": 0.182,
  "mean_abs_severity_distance": 0.545,
  "reference_recall": 0.591,
  "parse_failure_rate": 0.0
}


### Where does it fail?

Under-triage on an `EMERGENT` item is the finding that matters most — `CKM-003-M2`
(euglycemic DKA with a reassuring-looking glucose of 142) and `CKM-004-M2` (chest pressure
that resolved) are the two items designed to elicit it.

In [24]:
misses = results_full[~results_full["correct"]]
if len(misses) == 0:
    print("No misses.")
else:
    for _, row in misses.iterrows():
        direction = "UNDER-triaged" if row["under_triage"] else "OVER-triaged" if row["over_triage"] else "unparseable"
        print(f"{row['message_id']}  expected={row['expected']}  predicted={row['predicted']}  ({direction})")
        print(f"  ambiguity={row['ambiguity']}  requires_ehr={row['requires_ehr']}")
        print(f"  rationale: {row['rationale'][:220]}\n")

# Confusion matrix over the ordered levels.
order = sorted(TRIAGE_RANK, key=TRIAGE_RANK.get, reverse=True)
print("Confusion (rows = expected, cols = predicted):")
display(pd.crosstab(results_full["expected"], results_full["predicted"])
        .reindex(index=order, columns=order, fill_value=0))

CKM-001-M1  expected=URGENT  predicted=ROUTINE  (UNDER-triaged)
  ambiguity=high  requires_ehr=True
  rationale: The patient's symptoms of tingling in hands and heavy legs going up stairs are nonspecific and not immediately alarming. The patient's CKM stage is Stage 3, but there are no recent lab values indicating acute kidney inju

CKM-002-M2  expected=ROUTINE  predicted=URGENT  (OVER-triaged)
  ambiguity=moderate  requires_ehr=False
  rationale: The patient's home blood pressure reading of 150/95 mmHg exceeds the Stage 1 Hypertension threshold (130/80 to 139/89 mmHg) and is consistent with Stage 2 Hypertension [S2]. Given the patient's CKM stage is already Stage

CKM-003-M2  expected=EMERGENT  predicted=URGENT  (UNDER-triaged)
  ambiguity=high  requires_ehr=True
  rationale: The patient's report of shortness of breath (breathing kind of heavy and fast) is concerning for possible heart failure exacerbation, especially given their history of HFrEF [S3]. Additionally, abdominal pain cou

predicted,EMERGENT,URGENT,ROUTINE,SELF_CARE
expected,,,,
EMERGENT,0,2,0,0
URGENT,0,2,2,0
ROUTINE,0,1,3,0
SELF_CARE,0,0,1,0


## 16. Ablation — what do the EHR and the guideline actually contribute?

This is the probe's central experiment. Four conditions, same messages, same seed:

| Condition | EHR | RAG |
|---|---|---|
| Free text only (original baseline) | — | — |
| RAG only | — | ✓ |
| EHR only | ✓ | — |
| EHR + RAG | ✓ | ✓ |

The `requires_ehr=True` subset is where the EHR contribution should concentrate. If accuracy
on that subset does not move between "RAG only" and "EHR + RAG", the model is not using the
structured data — regardless of how good the headline number looks.

This runs 4 × 11 = 44 generations; expect several minutes.

In [25]:
conditions = [
    ("free text only", False, False),
    ("RAG only",       False, True),
    ("EHR only",       True,  False),
    ("EHR + RAG",      True,  True),
]

ablation_frames = []
for label, use_ehr, use_rag in conditions:
    print(f"\n=== {label} ===")
    df = evaluate(use_ehr=use_ehr, use_rag=use_rag, verbose=False)
    df["condition"] = label
    ablation_frames.append(df)
    print(json.dumps(summarize(df, label), indent=2))

ablation = pd.concat(ablation_frames, ignore_index=True)

print("\n=== Headline ===")
display(pd.DataFrame([summarize(d, d["condition"].iloc[0]) for d in ablation_frames]))

print("\n=== Accuracy on the requires_ehr subset (the EHR-dependent items) ===")
display(
    ablation[ablation["requires_ehr"]]
    .groupby("condition")["correct"].agg(["mean", "count"])
    .rename(columns={"mean": "accuracy", "count": "n"})
    .reindex([c[0] for c in conditions])
)

print("\n=== Accuracy by message ambiguity ===")
display(
    ablation.pivot_table(index="condition", columns="ambiguity", values="correct", aggfunc="mean")
    .reindex([c[0] for c in conditions])
)


=== free text only ===
{
  "condition": "free text only",
  "n": 11,
  "accuracy": 0.545,
  "under_triage_rate": 0.364,
  "over_triage_rate": 0.091,
  "mean_abs_severity_distance": 0.455,
  "reference_recall": 0.5,
  "parse_failure_rate": 0.0
}

=== RAG only ===
{
  "condition": "RAG only",
  "n": 11,
  "accuracy": 0.364,
  "under_triage_rate": 0.455,
  "over_triage_rate": 0.182,
  "mean_abs_severity_distance": 0.636,
  "reference_recall": 0.455,
  "parse_failure_rate": 0.0
}

=== EHR only ===
{
  "condition": "EHR only",
  "n": 11,
  "accuracy": 0.364,
  "under_triage_rate": 0.545,
  "over_triage_rate": 0.091,
  "mean_abs_severity_distance": 0.636,
  "reference_recall": 0.591,
  "parse_failure_rate": 0.0
}

=== EHR + RAG ===
{
  "condition": "EHR + RAG",
  "n": 11,
  "accuracy": 0.455,
  "under_triage_rate": 0.364,
  "over_triage_rate": 0.182,
  "mean_abs_severity_distance": 0.545,
  "reference_recall": 0.591,
  "parse_failure_rate": 0.0
}

=== Headline ===


,condition,n,accuracy,under_triage_rate,over_triage_rate,mean_abs_severity_distance,reference_recall,parse_failure_rate
0,free text only,11,0.545,0.364,0.091,0.455,0.500,0.0
1,RAG only,11,0.364,0.455,0.182,0.636,0.455,0.0
2,EHR only,11,0.364,0.545,0.091,0.636,0.591,0.0
3,EHR + RAG,11,0.455,0.364,0.182,0.545,0.591,0.0



=== Accuracy on the requires_ehr subset (the EHR-dependent items) ===


,accuracy,n
condition,,
free text only,0.571429,7
RAG only,0.428571,7
EHR only,0.285714,7
EHR + RAG,0.571429,7



=== Accuracy by message ambiguity ===


ambiguity,high,low,moderate
condition,,,
free text only,0.6,0.5,0.50
RAG only,0.4,0.5,0.25
EHR only,0.2,1.0,0.25
EHR + RAG,0.4,0.5,0.50


### Persist the run

Written with the condition, seed, and models attached, so a run stays interpretable after the
Colab session evaporates. Point `RESULTS_PATH` at Drive to keep it.

In [26]:
RESULTS_PATH = "/content/ckm_probe_results.csv"

ablation_out = ablation.copy()
ablation_out["chat_model"] = CHAT_MODEL
ablation_out["embed_model"] = EMBED_MODEL
ablation_out["seed"] = GEN_OPTIONS.get("seed")
ablation_out["corpus_id"] = CORPUS["corpus_id"]
ablation_out.to_csv(RESULTS_PATH, index=False)

print(f"Wrote {len(ablation_out)} rows to {RESULTS_PATH}")

Wrote 44 rows to /content/ckm_probe_results.csv


In [27]:
#!pip freeze > requirements.txt

In [28]:
# pip install -r requirements.txt for new notebook

---
## 17. Extending

**Add documents.** Ingestion is idempotent, so the guideline already in the store is not
re-embedded, and `answer()` and the evaluator immediately span everything ingested:

```python
ingest_documents([
    "/content/kdigo_ckd_guideline.pdf",
    "/content/ada_standards_of_care_2026.pdf",
])
```

**Add patients or messages.** Append to `ckm_patients.json` following the schema in Section 12.
`CASES` rebuilds on reload; no evaluator changes needed. Useful directions: multi-turn threads
(the corpus is single-turn today), messages where the *correct* answer is that the EHR is
stale, and non-English messages.

**Add a triage level.** Add it to `triage_levels` with a rank. `TRIAGE_RANK`, the prompt, and
`severity_distance` all derive from the JSON.

**Tuning**

| Symptom | Lever |
|---|---|
| Answers miss content you know is in the PDF | Raise `RETRIEVE_K`; try `mxbai-embed-large`; re-ingest |
| Answers vague / blend unrelated content | Lower `FINAL_K`; lower `CHUNK_TARGET_CHARS` |
| Retrieved chunks cut mid-argument | Raise `CHUNK_OVERLAP_CHARS` |
| Frequent `parse_failure_rate` | Add `"format": "json"` to `GEN_OPTIONS`; Ollama can constrain decoding |
| Model contradicts sources | Confirm `temperature: 0`; inspect retrieval first |

> Changing `EMBED_MODEL` or the chunking constants invalidates the index. Delete `CHROMA_DIR`
> and re-ingest, or the store will mix two embedding spaces and retrieval will quietly degrade.

**Two limits worth stating in any write-up.** With n=11 messages, a single flip moves accuracy
by 9 points — this corpus is sized to expose failure modes, not to support significance
testing; scale it before making comparative claims. And the expected labels are authored, not
adjudicated by multiple blinded clinicians, so they carry no inter-rater reliability estimate.
Both are fixable, and both are the kind of thing a reviewer asks about first.